In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

entity ------> care_Epi_contract_id

MPB-------------

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(ten.id AS varchar(100))

In [ ]:
SELECT
    care_epi_id,
    care_epi_src_id,
    care_epi_contr_id,
    z_src_system_id
FROM silver_care_episode
WHERE z_src_system_id = 'MPB'
  AND care_epi_contr_id IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    care_epi_id,
    care_epi_src_id,
    care_epi_contr_id,
    z_src_system_id,
    z_src_system_instance
FROM silver_care_episode
WHERE z_src_system_id = 'MPB'
  AND care_epi_contr_id IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    COUNT(*) AS total_mpb_rows,
    COUNT(care_epi_contr_id) AS populated_care_epi_contr_id_rows
FROM silver_care_episode
WHERE z_src_system_id = 'MPB';

In [ ]:
Updated MPB care_epi_contr_id mapping in the Care Episode notebook to use the new RDM Contracts table. The old source-derived contract key logic was replaced with a join to silver_rdm_contract using contr_src_sys_inst_id = 'MPB001' and contr_src_id = ten.id, and care_epi_contr_id is now populated from rdmc.contr_id as per the updated Monday definition.